# ARC research — her environment-play record

Reads `<data>/arc/results.jsonl` (one line per env attempt — local
practice, API, and official card tries alike) and the per-step
recordings under `<data>/arc/recordings/`. Run from the repo root;
GENESIS_DATA_DIR overrides the data dir.

In [ ]:
import json, os
from pathlib import Path

data = Path(os.environ.get('GENESIS_DATA_DIR', Path.home() / '.local/share/genesis'))
arc = data / 'arc'
rows = [json.loads(l) for l in (arc / 'results.jsonl').read_text().splitlines() if l.strip()] \
       if (arc / 'results.jsonl').exists() else []
print(f'{len(rows)} recorded env results')
rows[-3:] if rows else None

In [ ]:
# Per-game summary: tries, best level progress, wins
from collections import defaultdict
games = defaultdict(list)
for r in rows:
    games[r['game_id']].append(r)
summary = sorted(
    ((g, len(v), max(x['levels_completed'] for x in v),
      max((x['win_levels'] or 0) for x in v),
      sum(1 for x in v if x['state'] == 'WIN'),
      sum(1 for x in v if x['scorecard_id']))
     for g, v in games.items()), key=lambda t: t[0])
print(f'{"game":<16} {"tries":>5} {"lvls":>4} {"of":>3} {"wins":>4} {"official":>8}')
for g, n, lc, wl, wins, off in summary:
    print(f'{g:<16} {n:>5} {lc:>4} {wl:>3} {wins:>4} {off:>8}')

In [ ]:
# Learning curves — steps-per-completed-level per game, in order
for g, v in sorted(games.items()):
    seq = [(i, x['level_steps'], x['state']) for i, x in enumerate(v)]
    print(g, seq)

In [ ]:
# How the moves were made — priors vs learning, per game
for g, v in sorted(games.items()):
    src = defaultdict(int)
    for x in v:
        for k, n in x.get('decision_sources', {}).items():
            src[k] += n
    print(g, dict(src))

In [ ]:
# Raw recordings — one JSONL per run under recordings/
recs = sorted((arc / 'recordings').rglob('*.jsonl')) if (arc / 'recordings').is_dir() else []
print(len(recs), 'recording files')
for p in recs[-5:]:
    lines = p.read_text().splitlines()
    print(p.name, len(lines), 'events')